In [3]:
# Install (run once)
!pip install xgboost imbalanced-learn --quiet

# Imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import joblib

In [6]:
data = pd.read_csv("creditcard.csv")

print("Shape:", data.shape)
data.head()

Shape: (15936, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [7]:
# Remove ALL missing values
data = data.dropna().reset_index(drop=True)

# Ensure Class exists
if 'Class' not in data.columns:
    raise ValueError("❌ 'Class' column not found")

# Ensure correct type
data['Class'] = data['Class'].astype(int)

print("Cleaned Shape:", data.shape)
print(data['Class'].value_counts())

Cleaned Shape: (15935, 31)
Class
0    15862
1       73
Name: count, dtype: int64


In [8]:
# Safe Time handling
if 'Time' in data.columns:
    data['Hour'] = (data['Time'] // 3600) % 24
    data = data.drop(['Time'], axis=1)

In [9]:
X = data.drop('Class', axis=1)
y = data['Class']

# Ensure numeric data (IMPORTANT for XGBoost)
X = X.apply(pd.to_numeric, errors='coerce')

# Fill any leftover NaN
X = X.fillna(0)

print("Features shape:", X.shape)

Features shape: (15935, 30)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
scaler = StandardScaler()

# Copy to avoid pandas errors
X_train = X_train.copy()
X_test = X_test.copy()

if 'Amount' in X_train.columns:
    X_train.loc[:, 'Amount'] = scaler.fit_transform(X_train[['Amount']])
    X_test.loc[:, 'Amount'] = scaler.transform(X_test[['Amount']])

In [12]:
sm = SMOTE(random_state=42, k_neighbors=3)

X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Before:\n", y_train.value_counts())
print("After:\n", pd.Series(y_train_res).value_counts())

Before:
 Class
0    12690
1       58
Name: count, dtype: int64
After:
 Class
0    12690
1    12690
Name: count, dtype: int64


In [32]:
model = XGBClassifier(
    n_estimators=700,
    max_depth=10,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=25,
    random_state=42,
    eval_metric='logloss'
)
# Ensure float type (important)
X_train_res = X_train_res.astype(float)
X_test = X_test.astype(float)

model.fit(X_train, y_train)

print("✅ Model trained successfully")

✅ Model trained successfully


In [37]:
y_prob = model.predict_proba(X_test)[:, 1]

THRESHOLD = 0.01
y_pred = (y_prob >= THRESHOLD).astype(int)

In [38]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[3164    8]
 [   2   13]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3172
           1       0.62      0.87      0.72        15

    accuracy                           1.00      3187
   macro avg       0.81      0.93      0.86      3187
weighted avg       1.00      1.00      1.00      3187



In [39]:
joblib.dump(model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "features.pkl")

print("✅ Saved: model, scaler, features")

✅ Saved: model, scaler, features


In [40]:
print("Actual frauds:", sum(y_test))
print("Predicted frauds:", sum(y_pred))

Actual frauds: 15
Predicted frauds: 21


In [41]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[3164    8]
 [   2   13]]
